# Step 2: find roads near a GPS point

GPS points can drift away from the road. Before matching a whole route, we need to find nearby road segments and measure their distances in metres.

Run the cells from top to bottom. Change `trip_id` to try another training trip. This notebook uses `.nodes`, `.arcs`, and `.track`; candidate lookup does not need the reference `.route`.

If imports fail, run `%pip install pyproj "shapely>=2,<3"` in a notebook cell.

## 1. Read a trip

Nodes contain longitude and latitude. Each arc connects two node IDs. Each track row is a GPS point with a timestamp. Node IDs and arc IDs are row numbers starting at zero.

In [ ]:
from pathlib import Path
import pandas as pd
from pyproj import Transformer
from shapely.geometry import LineString, Point
from shapely.strtree import STRtree

trip_id = "00000005"
folder = Path("Student folder/train") / trip_id
nodes = pd.read_csv(folder / f"{trip_id}.nodes", sep=r"\s+", header=None, names=["lon", "lat"])
arcs = pd.read_csv(folder / f"{trip_id}.arcs", sep=r"\s+", header=None, names=["start", "end"])
track = pd.read_csv(folder / f"{trip_id}.track", sep=r"\s+", header=None, names=["lon", "lat", "time"])
track.head()

## 2. Convert degrees to metres

Choose a UTM zone from this trip's average longitude, then project both the road nodes and GPS points into the same coordinate system. Trips are in different regions, so we choose the zone separately for each trip.

`always_xy=True` keeps input coordinates in longitude, latitude order. The new `x` and `y` columns are in metres. UTM is a local approximation, suitable here for short trips.

In [ ]:
zone = int((track["lon"].mean() + 180) // 6) + 1
epsg = (32600 if track["lat"].mean() >= 0 else 32700) + zone
projection = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)

nodes["x"], nodes["y"] = projection.transform(nodes["lon"], nodes["lat"])
track["x"], track["y"] = projection.transform(track["lon"], track["lat"])
print("Projected coordinate system:", f"EPSG:{epsg}")
track.head()

## 3. Index the roads

Make one straight line per arc using its two endpoints. Keep lines in the original arc order, so a returned index is the original arc ID.

`STRtree` is an index that makes nearby-road searches faster.

In [ ]:
xy = nodes[["x", "y"]].to_numpy()
segments = [LineString(xy[[start, end]]) for start, end in arcs.itertuples(index=False, name=None)]
tree = STRtree(segments)
print("Road segments:", len(segments))

## 4. Find nearby roads

Change `point_id` to inspect another GPS point. `dwithin` returns roads whose actual line geometry is within `radius_m` metres of the point.

50 metres is a starting example, not a tuned threshold. An empty table means no road was found within that radius. Nearby roads are candidates; the closest road alone does not necessarily give the correct route.

In [ ]:
point_id = 0
radius_m = 50
point = Point(track.loc[point_id, ["x", "y"]])
arc_ids = tree.query(point, predicate="dwithin", distance=radius_m)

candidates = pd.DataFrame({
    "arc_id": arc_ids,
    "distance_m": [point.distance(segments[arc_id]) for arc_id in arc_ids],
})
candidates.sort_values("distance_m")

Later, repeat this lookup for each GPS point. The candidates become possible road states for route reconstruction; distances become model features.

References: [pyproj coordinate transformations](https://pyproj4.github.io/pyproj/stable/api/transformer.html), [Shapely STRtree](https://shapely.readthedocs.io/en/stable/strtree.html). File formats are described in the supplied Dataset Exploration guide.